<a href="https://colab.research.google.com/github/invi-bhagyesh/LLM_notebooks/blob/main/4_bit_LLM_Quantization_with_GPTQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!BUILD_CUDA_EXT=0 pip install -q auto-gptq transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.1/126.1 kB 9.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 128.6 MB/s eta 0:00:00


In [2]:
import random
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
from datasets import load_dataset
import torch
from transformers import AutoTokenizer
model = "gpt2"
out_dir = model+ "-GPTQ"


/usr/local/lib/python3.12/dist-packages/auto_gptq/nn_modules/triton_utils/kernels.py:410: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.12/dist-packages/auto_gptq/nn_modules/triton_utils/kernels.py:418: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.12/dist-packages/auto_gptq/nn_modules/triton_utils/kernels.py:461: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float16)


In [3]:
# load config for GPTQ
qat_config = BaseQuantizeConfig(
    bits = 4,
    damp_percent = 0.01, # damping applied during hessian based quantization, higher means stable quant but lower accuracy
    desc_act = False,# descending activation, consider dist of activations
    group_size = 128 # groups beign quantized together
)

tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoGPTQForCausalLM.from_pretrained(model, qat_config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
# load data
n_samples = 1024
data = load_dataset("allenai/c4", data_files = "en/c4-train.00001-of-01024.json.gz", split=f"train[:{n_samples*5}]")
tokenized_data = tokenizer("\n\n".join(data['text']), return_tensors="pt")


README.md: 0.00B [00:00, ?B/s]

en/c4-train.00001-of-01024.json.gz:   0%|          | 0.00/318M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2441065 > 1024). Running this sequence through the model will result in indexing errors


In [7]:
data

Dataset({
    features: ['text', 'timestamp', 'url'],
    num_rows: 5120
})

In [22]:
tokenized_data.input_ids.shape

torch.Size([1, 2441065])

In [23]:
tokenizer.model_max_length

1024

In [42]:
examples_ids = []
for _ in range(n_samples):
    i = random.randint(0, tokenized_data.input_ids.shape[1] - tokenizer.model_max_length - 1)
    j = i + tokenizer.model_max_length
    input_ids = tokenized_data.input_ids[:, i:j]
    attention_mask = torch.ones_like(input_ids)
    examples_ids.append({'input_ids': input_ids, 'attention_mask': attention_mask})

In [43]:
%%time

#Quantize with GPTQ
model.quantize(
    examples_ids,
    batch_size=1,
    use_triton=True,
)


TypeError: BaseGPTQForCausalLM.quantize.<locals>.LayerHijacker.forward() takes from 1 to 2 positional arguments but 7 were given

In [ ]:
model.save_quantized(out_dir, use_safetensors=True)
tokenizer.save_pretrained(out_dir)

In [45]:
device = "cude:0"

In [ ]:
model = AutoGPTQForCausalLM.from_pretrained(
    out_dir,
    device=device,
    use_safetensors=True,
    use_triton=True,
)
toknizer = AutoTokenizer.from_pretrained(out_dir)

In [ ]:
from transformers import pipeline
generator = pipeline("text-generation", model=model, toknizer=tokenizer)
generator("I have a dream", do_sample=True, max_length=50)[0]["generated_text"]